In [ ]:
with base as (

    select distinct
        ca.client_id,
        ca.campaign_name,

        case
            when ca.com_cus_sgr_desc like '%200%' then 200
            when ca.com_cus_sgr_desc like '%300%' then 300
            when ca.com_cus_sgr_desc like '%400%' then 400
            when ca.com_cus_sgr_desc like '%500%' then 500
        end as nominal

    from _ ca

    join _ cc
        on ca.client_id = cc.client_id

    where cc.campaigns_cnt >= 2

),

client_nominal as (

    select
        client_id,
        nominal,
        count(distinct campaign_name) as hit_cnt

    from base

    where nominal is not null

    group by
        client_id,
        nominal

),

client_max as (

    select
        client_id,
        max(hit_cnt) as max_hit_cnt

    from client_nominal

    group by client_id

)

select
    case
        when max_hit_cnt = 1 then 'все разные'
        else max_hit_cnt || ' раза в один номинал'
    end as group_name,

    count(*) as client_cnt,

    round(
        count(*) * 100.0 / sum(count(*)) over (),
        2
    ) as client_pct

from client_max

group by max_hit_cnt

order by max_hit_cnt;